# CUE 04: Timoni modules and bundles

Timoni packages CUE as modules with a lifecycle (build, apply, upgrade, rollback) and distributes them as OCI artifacts. `timoni mod init` scaffolds a module; `timoni build` renders it without a cluster.


In [ ]:
export HOME=/tmp
rm -rf /tmp/timoni-demo && mkdir -p /tmp/timoni-demo && timoni mod init demo /tmp/timoni-demo 2>&1 | tail -1 && find /tmp/timoni-demo/demo -type f -not -path '*/cue.mod/*' | sort


In [ ]:
export HOME=/tmp
timoni mod vet /tmp/timoni-demo/demo 2>&1 | tail -3 && timoni build demo /tmp/timoni-demo/demo -n lab | grep -E '^kind:|^  name:' | head -10


In [ ]:
export HOME=/tmp
timoni mod show config /tmp/timoni-demo/demo 2>&1 | head -25


The companion repository's `web` module is the same workload as every other renderer, with values files per environment validated against the module's `#Config`.


In [ ]:
export HOME=/tmp
mkdir -p /source/work && cd /source/work && [ -d gitops-renderers ] || git clone -q --recurse-submodules https://github.com/cznewt/gitops-renderers.git
cd /source/work/gitops-renderers && timoni build web examples/05-cue/timoni/web -n web -f examples/05-cue/timoni/values-prod.cue | yq 'select(.kind == "Deployment") | .spec.replicas, .metadata.labels'


In [ ]:
cd /source/work/gitops-renderers
printf 'values: ingress: className: "haproxy"\n' > /tmp/bad-values.cue && (timoni build web examples/05-cue/timoni/web -n web -f /tmp/bad-values.cue 2>&1 | grep -m1 -i 'conflicting\|error') || true


A bundle composes instances of modules for one cluster: local modules by path, published ones by OCI URL. `timoni bundle build` renders all of them at once.


In [ ]:
cd /source/work/gitops-renderers
cat > /tmp/bundle.cue <<'CUE'
bundle: {
    apiVersion: "v1alpha1"
    name:       "lab"
    instances: {
        podinfo: {
            module: url: "oci://ghcr.io/stefanprodan/modules/podinfo"
            namespace: "lab"
            values: {replicas: 2, ui: message: "hello from a bundle"}
        }
    }
}
CUE
export HOME=/tmp
timoni bundle build -f /tmp/bundle.cue | yq 'select(.kind == "Deployment") | .metadata.name + " replicas=" + (.spec.replicas | tostring)'
